In [151]:
# !pip install optuna

In [152]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit



In [153]:
df= pd.read_csv("labeled_pocket_features.csv")

In [154]:
df

,PDB_ID,Pocket_ID,Target,count_HIS,count_CYS,count_MET,count_TYR,aliphatic_hydrophobic_count,aromatic_count,propionate_stabilization_score,...,mean_alp._sph._solvent_access,apolar_alpha_sphere_proportion,hydrophobicity_score,volume_score,polarity_score,charge_score,proportion_of_polar_atoms,alpha_sphere_density,cent._of_mass___alpha_sphere_max_dist,flexibility
0,1DSO,Pocket 1,1,0,0,1,0,4,2,0,...,0.415,0.474,50.000,4.100,3.0,0.0,29.630,2.406,5.482,0.108
1,1DSO,Pocket 2,0,0,0,1,0,1,0,3,...,0.610,0.135,-6.200,4.500,8.0,1.0,53.846,3.892,9.113,0.052
2,1DSO,Pocket 3,0,1,0,0,1,3,1,1,...,0.517,0.171,4.538,3.615,8.0,2.0,41.379,3.767,8.308,0.195
3,1DSO,Pocket 4,0,1,0,0,0,1,0,1,...,0.480,0.233,0.000,3.071,9.0,0.0,44.828,4.557,11.606,0.319
4,1DSO,Pocket 5,0,0,0,0,0,0,1,3,...,0.672,0.792,-10.714,4.571,5.0,2.0,35.000,3.719,8.878,0.206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31822,3UQD,Pocket 2,0,0,0,1,0,7,0,3,...,0.543,0.267,7.767,3.267,14.0,2.0,44.681,11.287,30.805,0.252
31823,3UQD,Pocket 3,0,1,0,0,0,6,1,6,...,0.454,0.223,3.000,3.500,23.0,2.0,55.422,8.079,22.224,0.206
31824,3UQD,Pocket 4,0,0,0,1,0,6,0,2,...,0.425,0.448,9.963,3.259,12.0,1.0,40.909,7.401,17.666,0.098
31825,3UQD,Pocket 5,0,0,0,1,0,4,0,1,...,0.489,0.319,7.905,2.952,9.0,1.0,40.741,5.593,15.324,0.108


In [155]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [156]:
df_shuffled["Target"].value_counts()

,count
Target,
0,28707
1,3120


In [157]:
df_shuffled = df.drop_duplicates()

In [158]:
df_shuffled["Target"].value_counts()

,count
Target,
0,23744
1,2838


In [159]:
scale_pos_weight= df_shuffled["Target"].value_counts()[0]/df_shuffled["Target"].value_counts()[1]

In [160]:
scale_pos_weight # for model (imbalanced data)

np.float64(8.36645525017618)

In [161]:
# Save the IDs AND Center of Mass before dropping them so we can use them for AutoDock later!
df_ids = df[['PDB_ID', 'Pocket_ID', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z']]

In [162]:
df_shuffled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26582 entries, 0 to 31825
Data columns (total 35 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   PDB_ID                                 26582 non-null  object 
 1   Pocket_ID                              26582 non-null  object 
 2   Target                                 26582 non-null  int64  
 3   count_HIS                              26582 non-null  int64  
 4   count_CYS                              26582 non-null  int64  
 5   count_MET                              26582 non-null  int64  
 6   count_TYR                              26582 non-null  int64  
 7   aliphatic_hydrophobic_count            26582 non-null  int64  
 8   aromatic_count                         26582 non-null  int64  
 9   propionate_stabilization_score         26582 non-null  int64  
 10  has_CP_motif                           26582 non-null  int64  
 11  has_CXX

In [163]:
columns_to_drop = ['PDB_ID', 'Pocket_ID', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z']
df_cleaned = df_shuffled.drop(columns=columns_to_drop)

In [164]:
df_cleaned

,Target,count_HIS,count_CYS,count_MET,count_TYR,aliphatic_hydrophobic_count,aromatic_count,propionate_stabilization_score,has_CP_motif,has_CXXCH_motif,...,mean_alp._sph._solvent_access,apolar_alpha_sphere_proportion,hydrophobicity_score,volume_score,polarity_score,charge_score,proportion_of_polar_atoms,alpha_sphere_density,cent._of_mass___alpha_sphere_max_dist,flexibility
0,1,0,0,1,0,4,2,0,0,0,...,0.415,0.474,50.000,4.100,3.0,0.0,29.630,2.406,5.482,0.108
1,0,0,0,1,0,1,0,3,0,0,...,0.610,0.135,-6.200,4.500,8.0,1.0,53.846,3.892,9.113,0.052
2,0,1,0,0,1,3,1,1,0,0,...,0.517,0.171,4.538,3.615,8.0,2.0,41.379,3.767,8.308,0.195
3,0,1,0,0,0,1,0,1,0,0,...,0.480,0.233,0.000,3.071,9.0,0.0,44.828,4.557,11.606,0.319
4,0,0,0,0,0,0,1,3,0,0,...,0.672,0.792,-10.714,4.571,5.0,2.0,35.000,3.719,8.878,0.206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31821,0,1,0,0,1,11,3,7,0,0,...,0.460,0.215,13.045,3.750,25.0,2.0,54.054,9.448,28.614,0.179
31822,0,0,0,1,0,7,0,3,0,0,...,0.543,0.267,7.767,3.267,14.0,2.0,44.681,11.287,30.805,0.252
31823,0,1,0,0,0,6,1,6,0,0,...,0.454,0.223,3.000,3.500,23.0,2.0,55.422,8.079,22.224,0.206
31824,0,0,0,1,0,6,0,2,0,0,...,0.425,0.448,9.963,3.259,12.0,1.0,40.909,7.401,17.666,0.098


In [165]:
X = df_cleaned.drop(columns=['Target'])
y = df_cleaned['Target']

In [166]:
y

,Target
0,1
1,0
2,0
3,0
4,0
...,...
31821,0
31822,0
31823,0
31824,0


In [167]:
X

,count_HIS,count_CYS,count_MET,count_TYR,aliphatic_hydrophobic_count,aromatic_count,propionate_stabilization_score,has_CP_motif,has_CXXCH_motif,pocket_flatness_index,...,mean_alp._sph._solvent_access,apolar_alpha_sphere_proportion,hydrophobicity_score,volume_score,polarity_score,charge_score,proportion_of_polar_atoms,alpha_sphere_density,cent._of_mass___alpha_sphere_max_dist,flexibility
0,0,0,1,0,4,2,0,0,0,103.522,...,0.415,0.474,50.000,4.100,3.0,0.0,29.630,2.406,5.482,0.108
1,0,0,1,0,1,0,3,0,0,15.931,...,0.610,0.135,-6.200,4.500,8.0,1.0,53.846,3.892,9.113,0.052
2,1,0,0,1,3,1,1,0,0,13.018,...,0.517,0.171,4.538,3.615,8.0,2.0,41.379,3.767,8.308,0.195
3,1,0,0,0,1,0,1,0,0,1011.596,...,0.480,0.233,0.000,3.071,9.0,0.0,44.828,4.557,11.606,0.319
4,0,0,0,0,0,1,3,0,0,50.591,...,0.672,0.792,-10.714,4.571,5.0,2.0,35.000,3.719,8.878,0.206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31821,1,0,0,1,11,3,7,0,0,20.891,...,0.460,0.215,13.045,3.750,25.0,2.0,54.054,9.448,28.614,0.179
31822,0,0,1,0,7,0,3,0,0,99.675,...,0.543,0.267,7.767,3.267,14.0,2.0,44.681,11.287,30.805,0.252
31823,1,0,0,0,6,1,6,0,0,29.400,...,0.454,0.223,3.000,3.500,23.0,2.0,55.422,8.079,22.224,0.206
31824,0,0,1,0,6,0,2,0,0,113.072,...,0.425,0.448,9.963,3.259,12.0,1.0,40.909,7.401,17.666,0.098


In [168]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [191]:
import numpy as np
from scipy.special import expit

def focal_loss_lgb(y_true, y_pred, alpha=0.25, gamma=2.0):
    """
    Custom Focal Loss for LightGBM, vectorized for array inputs.
    y_pred: raw logit predictions from the model (array)
    y_true: actual labels (array of 0s and 1s)
    """
    # Convert raw logits to probabilities
    p = expit(y_pred)

    # Clip probabilities to avoid log(0) or log(1) issues for numerical stability
    epsilon = 1e-9
    p = np.clip(p, epsilon, 1 - epsilon)

    # Calculate base gradient (p - y_true)
    grad_base = p - y_true

    # Initialize gradient and hessian arrays
    grad = np.zeros_like(y_true, dtype=float)
    hess = np.zeros_like(y_true, dtype=float)

    # Identify positive (y_true == 1) and negative (y_true == 0) samples
    pos_mask = (y_true == 1)
    neg_mask = (y_true == 0)

    # --- Calculations for positive samples (y_true == 1) ---
    term1_pos = (1 - p[pos_mask]) ** gamma

    grad[pos_mask] = grad_base[pos_mask] * term1_pos * alpha

    # Hessian for positive class
    hess[pos_mask] = p[pos_mask] * (1 - p[pos_mask]) * alpha * (
        term1_pos - gamma * (1 - p[pos_mask])**(gamma - 1) * p[pos_mask] * np.log(p[pos_mask])
    )

    # --- Calculations for negative samples (y_true == 0) ---
    term2_neg = p[neg_mask] ** gamma

    grad[neg_mask] = grad_base[neg_mask] * term2_neg * (1 - alpha)

    # Hessian for negative class
    hess[neg_mask] = p[neg_mask] * (1 - p[neg_mask]) * (1 - alpha) * (
        term2_neg + gamma * p[neg_mask]**(gamma - 1) * (1 - p[neg_mask]) * np.log(1 - p[neg_mask])
    )

    return grad, np.abs(hess) # Return absolute value of hessian for stability

# Wrapper function so LightGBM can use it during training
def focal_loss_wrapper(preds, train_data):
    labels = train_data.get_label()
    # You can let Optuna tune these, but 0.25 and 2.0 are industry standards
    grad, hess = focal_loss_lgb(labels, preds, alpha=0.25, gamma=2.0)
    return grad, hess

In [197]:
def objective(trial):
    # 1. Let Optuna smartly suggest parameters
    param = {
        'verbosity': -1, # Keeps the logs clean
        'boosting_type': 'gbdt',
        'objective': focal_loss_wrapper, # Tell LightGBM to use Focal Loss
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 80),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 3.0, 8.0)
    }

    # Extract n_estimators separately
    optuna_rounds = trial.suggest_int('n_estimators', 100, 500)

    # Cross-validation setup (5-fold)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_va, label=y_va)

        # 2. Train model WITH FOCAL LOSS
        gbm = lgb.train(
            param,
            dtrain,
            num_boost_round=optuna_rounds,
            valid_sets=[dval]
            # FIXED: Removed the early_stopping callback that was causing the crash
        )

        # Predict on validation fold
        raw_logits = gbm.predict(X_va)
        preds = expit(raw_logits)


        binary_preds = (preds >= 0.50).astype(int) # using normal threshold

        # Calculate F1 for this fold
        fold_f1 = f1_score(y_va, binary_preds)
        f1_scores.append(fold_f1)

    return np.mean(f1_scores)

In [198]:
import optuna

print("Starting Optuna Bayesian Optimization with Focal Loss...")
# We want to MAXIMIZE the F1 score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"\nOptimization Complete!")
print(f"Best F1 Score Achieved: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-18 19:40:08,094] A new study created in memory with name: no-name-f92836be-9748-48e0-b871-d912127b0829


Starting Optuna Bayesian Optimization with Focal Loss...


[I 2026-06-18 19:40:21,164] Trial 0 finished with value: 0.3448270392520411 and parameters: {'learning_rate': 0.011467088625334156, 'num_leaves': 72, 'max_depth': 6, 'min_child_samples': 15, 'subsample': 0.8591226996946009, 'colsample_bytree': 0.9016583575716044, 'reg_alpha': 0.21668898756317911, 'reg_lambda': 0.9730939275248965, 'scale_pos_weight': 7.763193725778387, 'n_estimators': 250}. Best is trial 0 with value: 0.3448270392520411.
[I 2026-06-18 19:40:45,271] Trial 1 finished with value: 0.4677762950900878 and parameters: {'learning_rate': 0.010273567443476555, 'num_leaves': 44, 'max_depth': 9, 'min_child_samples': 43, 'subsample': 0.6439899176412359, 'colsample_bytree': 0.6777654803669642, 'reg_alpha': 0.017311771561752565, 'reg_lambda': 0.5665758100023254, 'scale_pos_weight': 3.806271493637443, 'n_estimators': 497}. Best is trial 1 with value: 0.4677762950900878.
[I 2026-06-18 19:40:54,710] Trial 2 finished with value: 0.3441616497736131 and parameters: {'learning_rate': 0.02229


Optimization Complete!
Best F1 Score Achieved: 0.5942
Best Parameters:
  learning_rate: 0.09837431632875772
  num_leaves: 46
  max_depth: 12
  min_child_samples: 56
  subsample: 0.656293158938368
  colsample_bytree: 0.7647658500797352
  reg_alpha: 0.1426117524723997
  reg_lambda: 0.2910550666866694
  scale_pos_weight: 6.920211609252363
  n_estimators: 431


In [203]:
from sklearn.metrics import classification_report, accuracy_score

best_params = study.best_params

# FIXED: Inject the focal loss wrapper into the best parameters dictionary
best_params['objective'] = focal_loss_wrapper

# Extract the tuned tree count to use in num_boost_round
final_rounds = best_params.pop('n_estimators', 150)

print("Training final robust model with optimized parameters...")
dtrain_final = lgb.Dataset(X_train, label=y_train)

# Train the final model
final_optimized_model = lgb.train(
    best_params,
    dtrain_final,
    num_boost_round=final_rounds # FIXED: Uses the optimized tree count
)

raw_logits_test = final_optimized_model.predict(X_test)
y_probs_test = expit(raw_logits_test) # Convert logits to probabilities!


# optimal_threshold =[0.20,0.25,0.30,0.35,0.4,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80]
# for t in optimal_threshold:
#     y_pred_final = (y_probs_test >= t).astype(int)
#     print(f"Threshold: {t}")
#     print(classification_report(y_test, y_pred_final))
#     print(f"Final Model Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
#     print("\n--- Final Focal Loss Classification Report ---")


optimal_threshold = 0.35
y_pred_final = (y_probs_test >= optimal_threshold).astype(int)
print(f"Threshold: {optimal_threshold}")
print(classification_report(y_test, y_pred_final))




# Print Final Results


Training final robust model with optimized parameters...
Threshold: 0.35
              precision    recall  f1-score   support

           0       0.96      0.97      0.96      4777
           1       0.70      0.61      0.65       540

    accuracy                           0.93      5317
   macro avg       0.83      0.79      0.81      5317
weighted avg       0.93      0.93      0.93      5317

